In [4]:
import requests
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from typing import List, Dict, Any
from datetime import datetime
import os
from tqdm import tqdm
from typing import List
from sklearn.metrics import ndcg_score
from huggingface_hub import InferenceClient

In [ ]:
API_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"
HEADERS = {"Authorization": f"Bearer {API_TOKEN}"}

In [7]:
def get_teacher_embedding(text):
    """Get embeddings from teacher model via Hugging Face API"""
    response = requests.post(
        "https://api-inference.huggingface.co/models/intfloat/multilingual-e5-large-instruct",
        headers=HEADERS,
        json={"inputs": text}
    )
    response.raise_for_status()
    return response.json()[0]  

In [8]:
def load_scifact_data(file_path):
    """Load test.tsv from SciFact dataset (query-id, corpus-id, score)"""
    return pd.read_csv("test.tsv", sep="\t")

In [ ]:
    "QDRANT_URL = "http://localhost:6333"  # Remove trailing slash
",
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    timeout=60.0  # Add timeout
)

In [13]:
from qdrant_client import models

COLLECTION_NAME = "scifact"
qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
)

C:\Users\BeloAbhigyan\AppData\Local\Temp\ipykernel_11760\2775752759.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

In [ ]:
def generate_teacher_embeddings(data):
    """Generate embeddings for all queries and store in Qdrant"""
    for _, row in tqdm(data.iterrows(), total=len(data)):
        query_text = f"query-{row['query-id']}"  # Query ID as text
        embedding = get_teacher_embedding(query_text)  # Call teacher model API
        store_embedding(row["query-id"], embedding)